<a href="https://colab.research.google.com/github/rushisarena/NLP-BN-studentRequestApproval/blob/main/NLP%2BBN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pgmpy
import re
import numpy as np
import pandas as pd
from pgmpy.models import DiscreteBayesianNetwork  # or BayesianNetwork in newer pgmpy
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 45.6 MB/s eta 0:00:00


In [2]:
PRIORITY_KEYWORDS = {
    "HIGH": [
        "illness", "sick", "fever", "hospital", "accident", "emergency",
        "surgery", "medical", "injured", "injury", "icu", "critical",
        "death", "funeral", "bereavement", "covid",
    ],
    "MEDIUM": [
        "project", "hackathon", "presentation", "conference", "exam",
        "internship", "workshop", "seminar", "competition", "academic",
        "research", "thesis", "viva",
    ],
    "LOW": [
        "marriage", "wedding", "function", "festival", "family event",
        "vacation", "trip", "birthday", "personal",
    ],
}

In [3]:
# Credibility signals
PROOF_POSITIVE = [
    "letter", "certificate", "document", "proof", "prescription",
    "slip", "invitation", "admit card", "permission", "confirmed",
    "verified", "hospital slip", "medical certificate",
]
PROOF_NEGATIVE = [
    "no letter", "no proof", "no document", "don't have", "without letter",
    "no certificate", "lost", "forgot", "can't provide", "unable to provide",
]
PARENT_POSITIVE = ["parents approved", "parent approved", "parents have approved",
                   "parents gave", "parents gave permission", "parents agreed",
                   "parent letter", "parents' letter", "parents supporting",
                   "parents are supporting"]
PARENT_NEGATIVE = ["parents don't know", "parents not aware",
                   "without parents", "parents not informed"]
VAGUE_PATTERNS   = ["some reason", "personal reason", "urgent work",
                    "some work", "cannot say", "confidential"]

In [4]:
def classify_priority(text: str) -> tuple[str, float]:
    """Return (priority_label, reason_score)."""
    text_lower = text.lower()
    for level in ("HIGH", "MEDIUM", "LOW"):
        for kw in PRIORITY_KEYWORDS[level]:
            if kw in text_lower:
                scores = {"HIGH": 0.9, "MEDIUM": 0.65, "LOW": 0.3}
                return level, scores[level]
    return "LOW", 0.2          # default: unknown → treated as low


In [5]:
def assess_credibility(text: str) -> tuple[float, list[str]]:
    """
    Return (credibility_score ∈ [0,1], list_of_reasons_for_score).
    Implements Rules A / B / C from the spec.
    """
    text_lower = text.lower()
    score  = 0.5          # neutral baseline
    notes  = []

    # ── positive signals ──────────────────────────────────────────────────────
    has_parent_positive = any(p in text_lower for p in PARENT_POSITIVE)
    has_proof_positive  = any(p in text_lower for p in PROOF_POSITIVE)

    # Rule B: strong proof + parent approval → credibility override
    if has_parent_positive and has_proof_positive:
        score += 0.4
        notes.append(" Parent approval + documentary proof → credibility override (Rule B)")
    elif has_parent_positive:
        score += 0.1
        notes.append("📋 Parent approval claimed (no proof yet)")
    elif has_proof_positive:
        score += 0.25
        notes.append("📄 Some documentary proof mentioned")

    # ── negative signals (Rule C) ─────────────────────────────────────────────
    has_proof_negative  = any(p in text_lower for p in PROOF_NEGATIVE)
    has_parent_negative = any(p in text_lower for p in PARENT_NEGATIVE)
    is_vague            = any(v in text_lower for v in VAGUE_PATTERNS)

    if has_proof_negative:
        score -= 0.3
        notes.append("  Explicitly no proof/letter → trust reduced (Rule C)")
    if has_parent_negative:
        score -= 0.2
        notes.append("  Parents unaware or not informed → trust reduced")
    if is_vague:
        score -= 0.15
        notes.append("  Vague justification → trust penalised (Rule C)")

    # ── contradiction check ───────────────────────────────────────────────────
    if has_parent_positive and has_parent_negative:
        score -= 0.3
        notes.append(" Contradictory parent statements → very low credibility (Rule C)")

    # Pattern: parent approved BUT no letter  →  low credibility (Rule A / C)
    if has_parent_positive and has_proof_negative:
        score -= 0.25
        notes.append("  Parents approved but no letter provided → pattern-based distrust (Rule A/C)")

    score = float(np.clip(score, 0.0, 1.0))
    if not notes:
        notes.append("   No strong credibility signals detected; using baseline")
    return score, notes

In [6]:
def extract_nlp_features(text: str) -> dict:
    """Master NLP extraction — returns all features needed by the scoring layer."""
    priority, reason_score     = classify_priority(text)
    credibility_score, cr_notes = assess_credibility(text)
    return {
        "raw_text":          text,
        "priority":          priority,
        "reason_score":      round(reason_score, 3),
        "credibility_score": round(credibility_score, 3),
        "credibility_notes": cr_notes,
    }



In [7]:
# 2.  SCORING → BAYESIAN VARIABLE MAPPING


def map_to_bn_evidence(features: dict) -> tuple[dict, list[str]]:
    """
    Convert continuous NLP scores to discrete BN evidence variables.
    Returns (evidence_dict, list_of_mapping_notes).

    BN variables: SRW, FP, PR, HODP, PIP, OB  (all binary 0/1)
    We set evidence only for the root / input nodes (SRW, FP, PR);
    inference computes the rest.
    """
    rs  = features["reason_score"]
    cs  = features["credibility_score"]
    pri = features["priority"]
    notes = []

    # ── SRW: student request weight ──────────────────────────────────────────
    srw = 1 if rs >= 0.5 else 0
    notes.append(f"SRW={'1 (strong)' if srw else '0 (weak)'} "
                 f"← reason_score={rs} (priority={pri})")

    # ── PR: parent response credibility ──────────────────────────────────────
    if cs >= 0.65:
        pr = 1
        notes.append(f"PR=1 (trusted) ← credibility={cs:.2f} ≥ 0.65")
    elif cs >= 0.4:
        # soft boundary — inject weak PR=1 (will be dominated by CPT priors)
        pr = 1
        notes.append(f"PR=1 (marginal) ← credibility={cs:.2f} in [0.4, 0.65); "
                     f"uncertainty preserved via CPT")
    else:
        pr = 0
        notes.append(f"PR=0 (untrusted) ← credibility={cs:.2f} < 0.4 "
                     f"→ pattern-based distrust applied")

    # ── FP: faculty permission ────────────────────────────────────────────────
    # Rule A: LOW priority + LOW credibility → bias FP toward 0
    if pri == "LOW" and cs < 0.45:
        fp = 0
        notes.append("FP=0 ← Rule A: LOW priority + LOW credibility "
                     "→ faculty unlikely to approve")
    elif pri == "HIGH":
        fp = 1
        notes.append("FP=1 ← HIGH priority (medical/emergency) → faculty approves")
    elif pri == "MEDIUM":
        fp = 1 if cs >= 0.5 else 0
        notes.append(f"FP={fp} ← MEDIUM priority, credibility={cs:.2f}")
    else:   # LOW priority but credibility acceptable
        fp = 1 if cs >= 0.6 else 0
        notes.append(f"FP={fp} ← LOW priority; credibility={cs:.2f} threshold 0.6")

    evidence = {"SRW": srw, "FP": fp, "PR": pr}
    return evidence, notes


In [8]:
# 3.  BAYESIAN NETWORK  (pgmpy)

# Edge list (mandatory structure from spec)
EDGES = [
    ("SRW", "FP"),
    ("SRW", "PR"),
    ("FP",  "HODP"),
    ("PR",  "HODP"),
    ("HODP","PIP"),
    ("PR",  "PIP"),
    ("PIP", "OB"),
]

def generate_synthetic_data(n: int = 4000, seed: int = 42) -> pd.DataFrame:
    """
    Generates realistic synthetic training data for model.fit().
    CPT assumptions reflect real-world institutional patterns.
    """
    rng = np.random.default_rng(seed)

    def coin(p, size=n):
        return (rng.random(size) < p).astype(int)

    SRW  = coin(0.55)          # 55 % of requests are 'strong'

    # Faculty is more likely to approve strong requests
    FP   = np.where(SRW == 1, coin(0.72), coin(0.30))

    # Parent response depends on strength of request
    PR   = np.where(SRW == 1, coin(0.65), coin(0.25))

    # HoD needs BOTH faculty and parent backing
    hodp_p = 0.1 * np.ones(n)
    hodp_p = np.where((FP == 1) & (PR == 1), 0.85, hodp_p)
    hodp_p = np.where((FP == 1) & (PR == 0), 0.45, hodp_p)
    hodp_p = np.where((FP == 0) & (PR == 1), 0.35, hodp_p)
    HODP  = coin(hodp_p)

    # Principal looks at HoD + parent
    pip_p = 0.05 * np.ones(n)
    pip_p = np.where((HODP == 1) & (PR == 1), 0.88, pip_p)
    pip_p = np.where((HODP == 1) & (PR == 0), 0.52, pip_p)
    pip_p = np.where((HODP == 0) & (PR == 1), 0.28, pip_p)
    PIP   = coin(pip_p)

    # Final outcome depends on Principal
    OB = np.where(PIP == 1, coin(0.90), coin(0.08))

    return pd.DataFrame(
        {"SRW": SRW, "FP": FP, "PR": PR, "HODP": HODP, "PIP": PIP, "OB": OB}
    )


def build_and_fit_model() -> tuple:
    """Build the BN, fit on synthetic data, return (model, inference_engine)."""
    try:
        model = DiscreteBayesianNetwork(EDGES)
    except Exception:
        # Fallback for older pgmpy versions
        from pgmpy.models import BayesianNetwork
        model = BayesianNetwork(EDGES)

    data = generate_synthetic_data()
    model.fit(data, estimator=MaximumLikelihoodEstimator)
    inference = VariableElimination(model)
    return model, inference

In [9]:
# 4.  FULL PIPELINE


def evaluate_request(text: str, inference_engine) -> dict:
    """
    End-to-end pipeline:
      text → NLP → scoring → BN evidence → probabilistic inference → result
    """
    # Step 1: NLP
    features = extract_nlp_features(text)

    # Step 2: Score → BN evidence
    evidence, mapping_notes = map_to_bn_evidence(features)

    # Step 3: Bayesian inference
    result = inference_engine.query(variables=["OB"], evidence=evidence,
                                    show_progress=False)
    approval_prob = float(result.values[1])   # P(OB=1)

    # Step 4: Assemble explanation
    credibility_level = (
        "HIGH"   if features["credibility_score"] >= 0.65 else
        "MEDIUM" if features["credibility_score"] >= 0.4  else
        "LOW"
    )
    override_triggered = any("Rule B" in n for n in features["credibility_notes"])
    distrust_applied   = any("Rule A" in n or "Rule C" in n
                             for n in features["credibility_notes"])

    return {
        "approval_probability":  round(approval_prob, 4),
        "priority":              features["priority"],
        "reason_score":          features["reason_score"],
        "credibility_score":     features["credibility_score"],
        "credibility_level":     credibility_level,
        "evidence_injected":     evidence,
        "override_triggered":    override_triggered,
        "distrust_applied":      distrust_applied,
        "credibility_notes":     features["credibility_notes"],
        "mapping_notes":         mapping_notes,
        "key_variables":         _identify_key_variables(evidence),
    }


def _identify_key_variables(evidence: dict) -> list[str]:
    """Heuristically identify which variables drove the outcome most."""
    key = []
    if evidence.get("PR") == 1:
        key.append("PR (parent response) — boosted approval")
    elif evidence.get("PR") == 0:
        key.append("PR (parent response) — suppressed approval")
    if evidence.get("FP") == 1:
        key.append("FP (faculty permission) — boosted via HoD")
    elif evidence.get("FP") == 0:
        key.append("FP (faculty permission) — suppressed via HoD → PIP")
    if evidence.get("SRW") == 1:
        key.append("SRW (request weight) — high-priority uplift")
    return key


def print_report(text: str, result: dict):
    """Pretty-print a full decision report."""
    sep = "─" * 60
    print(f"\n{sep}")
    print(f" REQUEST: {text}")
    print(sep)
    print(f" Approval Probability : {result['approval_probability']:.1%}")
    print()
    print(f" Priority              : {result['priority']}  "
          f"(reason_score={result['reason_score']})")
    print(f" Credibility           : {result['credibility_level']}  "
          f"(score={result['credibility_score']})")
    print(f" Override (Rule B)     : {'YES' if result['override_triggered'] else 'NO'}")
    print(f"  Distrust (Rule A/C)  : {'YES' if result['distrust_applied'] else 'NO'}")
    print()
    print(" Credibility Analysis:")
    for note in result["credibility_notes"]:
        print(f"   {note}")
    print()
    print(" BN Evidence Injected:")
    for note in result["mapping_notes"]:
        print(f"   {note}")
    print()
    print(" Key Decision Variables:")
    for kv in result["key_variables"]:
        print(f"   • {kv}")
    print(sep)


In [10]:
# 5.  TEST CASES


TEST_CASES = {
    "Medical Emergency (HIGH priority)": (
        "I need urgent leave as I met with a road accident yesterday. "
        "I am currently in the hospital and have a medical certificate from the doctor. "
        "My parents have also sent a letter confirming this."
    ),
    "Marriage – No Proof (LOW credibility)": (
        "I need leave because I want to attend a marriage ceremony. "
        "My parents have approved but I don't have any letter or document to provide."
    ),
    "Marriage – Strong Proof (Override)": (
        "I am requesting leave to attend my sister's wedding next week. "
        "I have a formal invitation letter and my parents have also sent a written permission letter "
        "confirming they approve of this absence."
    ),
    "Academic Hackathon (MEDIUM priority)": (
        "I have been selected for a national-level hackathon at IIT Bombay. "
        "I have the official invitation document. My parents are supporting my participation."
    ),
    "Vague / No Proof (Very Low credibility)": (
        "I need leave for some personal reason. I can't really explain it right now. "
        "My parents are not aware yet. I don't have any documents."
    ),
}





In [11]:

# 6.  MAIN


if __name__ == "__main__":
    print("Building and fitting Bayesian Network …")
    model, inference = build_and_fit_model()
    print(" Model ready.\n")

    # Interactive mode
    print("=" * 60)
    print("  INSTITUTIONAL LEAVE APPROVAL SYSTEM")
    print("  Hybrid NLP + Bayesian Network")
    print("=" * 60)

    print("\n--- Running built-in test cases ---\n")
    for label, text in TEST_CASES.items():
        print(f"\n[TEST CASE] {label}")
        result = evaluate_request(text, inference)
        print_report(text, result)

Building and fitting Bayesian Network …
 Model ready.

  INSTITUTIONAL LEAVE APPROVAL SYSTEM
  Hybrid NLP + Bayesian Network

--- Running built-in test cases ---


[TEST CASE] Medical Emergency (HIGH priority)

────────────────────────────────────────────────────────────
 REQUEST: I need urgent leave as I met with a road accident yesterday. I am currently in the hospital and have a medical certificate from the doctor. My parents have also sent a letter confirming this.
────────────────────────────────────────────────────────────
 Approval Probability : 72.3%

 Priority              : HIGH  (reason_score=0.9)
 Credibility           : HIGH  (score=0.75)
 Override (Rule B)     : NO
  Distrust (Rule A/C)  : NO

 Credibility Analysis:
   📄 Some documentary proof mentioned

 BN Evidence Injected:
   SRW=1 (strong) ← reason_score=0.9 (priority=HIGH)
   PR=1 (trusted) ← credibility=0.75 ≥ 0.65
   FP=1 ← HIGH priority (medical/emergency) → faculty approves

 Key Decision Variables:
   • PR (par

In [13]:
    print("\n\n--- Interactive Mode ---")
    while True:
        user_input = input("\nEnter student request (or 'quit'): ").strip()
        if user_input.lower() in ("quit", "exit", "q"):
            break
        result = evaluate_request(user_input, inference)
        print_report(user_input, result)



--- Interactive Mode ---

Enter student request (or 'quit'): quit
